# 📱 Deteksi Kecanduan Smartphone
## Notebook 4: Evaluasi, Prediksi & Visualisasi Model
---
Input  : `xgboost_model.pkl`, `X_test.csv`, `y_test.csv`, `X_train.csv`, `y_train.csv`  
Output : Laporan evaluasi lengkap + visualisasi

### 4.1 Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
from sklearn.preprocessing import label_binarize
from sklearn.inspection import permutation_importance
import shap

plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('✅ Library berhasil diimport')

### 4.2 Load Model & Data

In [ ]:
model   = joblib.load('xgboost_model.pkl')
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test  = pd.read_csv('y_test.csv').squeeze()

# Konversi ke 0-indexed
y_train_0 = y_train - 1
y_test_0  = y_test  - 1

y_pred    = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

CLASS_NAMES = ['Sangat Rendah (1)', 'Rendah (2)', 'Sedang (3)', 'Tinggi (4)', 'Sangat Tinggi (5)']

print(f'Model loaded: {type(model).__name__}')
print(f'Test samples : {len(y_test)}')

### 4.3 Metrik Evaluasi Lengkap

In [ ]:
acc  = accuracy_score(y_test_0, y_pred)
prec = precision_score(y_test_0, y_pred, average='weighted')
rec  = recall_score(y_test_0, y_pred, average='weighted')
f1   = f1_score(y_test_0, y_pred, average='weighted')

# ROC-AUC (One-vs-Rest)
y_bin = label_binarize(y_test_0, classes=[0,1,2,3,4])
auc   = roc_auc_score(y_bin, y_pred_proba, multi_class='ovr', average='weighted')

print('=' * 50)
print('   LAPORAN EVALUASI MODEL – XGBoost')
print('=' * 50)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}  ({prec*100:.2f}%)')
print(f'  Recall    : {rec:.4f}  ({rec*100:.2f}%)')
print(f'  F1-Score  : {f1:.4f}  ({f1*100:.2f}%)')
print(f'  ROC-AUC   : {auc:.4f}  ({auc*100:.2f}%)')
print('=' * 50)

# Classification report per kelas
print('\n=== Classification Report per Kelas ===')
print(classification_report(y_test_0, y_pred, target_names=CLASS_NAMES))

### 4.4 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test_0, y_pred)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Confusion Matrix (Count)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=range(1, 6), yticklabels=range(1, 6),
            linewidths=0.5, linecolor='gray', cbar_kws={'shrink': 0.8})
axes[0].set_title('Confusion Matrix (Jumlah)', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')

# --- Confusion Matrix (Persentase)
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Greens', ax=axes[1],
            xticklabels=range(1, 6), yticklabels=range(1, 6),
            linewidths=0.5, linecolor='gray', cbar_kws={'shrink': 0.8, 'label': '%'})
axes[1].set_title('Confusion Matrix (Persentase %)', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Prediksi')
axes[1].set_ylabel('Aktual')

plt.suptitle('Analisis Confusion Matrix – XGBoost Smartphone Addiction Detection',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('04_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix disimpan.')

### 4.5 Feature Importance (XGBoost Bawaan)

In [ ]:
importance_types = ['weight', 'gain', 'cover']
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, imp_type in zip(axes, importance_types):
    scores = model.get_booster().get_score(importance_type=imp_type)
    feat_df = pd.DataFrame(list(scores.items()), columns=['Fitur', 'Score'])
    feat_df = feat_df.sort_values('Score', ascending=True)

    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(feat_df)))
    ax.barh(feat_df['Fitur'], feat_df['Score'], color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(f'Feature Importance\n({imp_type.capitalize()})', fontweight='bold')
    ax.set_xlabel('Score')

plt.suptitle('XGBoost Feature Importance – 3 Metode', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('04_feature_importance_xgb.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.6 Permutation Feature Importance

In [ ]:
perm_imp = permutation_importance(model, X_test, y_test_0,
                                   n_repeats=20, random_state=42, n_jobs=-1)

perm_df = pd.DataFrame({
    'Fitur'      : X_test.columns,
    'Importance' : perm_imp.importances_mean,
    'Std'        : perm_imp.importances_std
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('=== Permutation Feature Importance ===')
print(perm_df.to_string(index=False))

plt.figure(figsize=(12, 7))
colors = ['#e74c3c' if v > 0.01 else '#95a5a6' for v in perm_df['Importance']]
plt.barh(perm_df['Fitur'][::-1], perm_df['Importance'][::-1],
         xerr=perm_df['Std'][::-1], color=colors[::-1],
         edgecolor='black', linewidth=0.5, capsize=3)
plt.xlabel('Penurunan Akurasi (Mean Importance)')
plt.title('Permutation Feature Importance – XGBoost\nDeteksi Kecanduan Smartphone', fontweight='bold')
plt.tight_layout()
plt.savefig('04_permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.7 SHAP Values – Interpretasi Model

In [ ]:
explainer  = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# SHAP Summary Plot (Bar)
plt.figure(figsize=(12, 7))
shap.summary_plot(shap_values, X_test, plot_type='bar',
                  class_names=CLASS_NAMES, show=False, max_display=15)
plt.title('SHAP Feature Importance (Bar) – XGBoost', fontweight='bold')
plt.tight_layout()
plt.savefig('04_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# SHAP Beeswarm Plot (kelas 4 = Tinggi, index 3)
plt.figure(figsize=(12, 7))
shap.summary_plot(shap_values[3], X_test, show=False, max_display=15)
plt.title('SHAP Beeswarm – Kelas 4 (Tinggi)', fontweight='bold')
plt.tight_layout()
plt.savefig('04_shap_beeswarm_kelas4.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP plot disimpan.')

### 4.8 Prediksi Sampel Baru

In [ ]:
# Contoh prediksi untuk 5 sampel dari test set
selected_features = joblib.load('selected_features.pkl')
scaler            = joblib.load('minmax_scaler.pkl')

sample_indices = [0, 5, 20, 40, 60]
samples = X_test.iloc[sample_indices]
actual  = y_test_0.iloc[sample_indices].values
pred    = model.predict(samples)
prob    = model.predict_proba(samples)

print('=== Prediksi Sampel Test ===')
print(f'{"No":>4} | {"Aktual":>8} | {"Prediksi":>9} | {"Probabilitas Max":>18} | Status')
print('-' * 65)
for i, (act, prd, prb) in enumerate(zip(actual, pred, prob)):
    status = '✅ Benar' if act == prd else '❌ Salah'
    max_prob = max(prb)
    print(f'{i+1:>4} | Kelas {act+1:>2}  | Kelas {prd+1:>2}    | {max_prob:>17.2%} | {status}')

### 4.9 Visualisasi Prediksi vs Aktual

In [ ]:
# Scatter plot aktual vs prediksi (jitter untuk visibilitas)
np.random.seed(42)
jitter = np.random.uniform(-0.15, 0.15, len(y_test_0))

colors_map = {0: '#2ecc71', 1: '#3498db', 2: '#f39c12', 3: '#e74c3c', 4: '#8e44ad'}
point_colors = [colors_map[int(v)] for v in y_test_0]

plt.figure(figsize=(10, 8))
plt.scatter(y_test_0 + jitter, y_pred + jitter,
            c=point_colors, alpha=0.6, s=40, edgecolors='none')
plt.plot([-0.5, 4.5], [-0.5, 4.5], 'k--', linewidth=2, label='Perfect Prediction')
plt.xticks(range(5), [f'Kelas {i+1}' for i in range(5)])
plt.yticks(range(5), [f'Kelas {i+1}' for i in range(5)])
plt.xlabel('Nilai Aktual')
plt.ylabel('Nilai Prediksi')
plt.title('Scatter Plot – Prediksi vs Aktual\nXGBoost Smartphone Addiction Detection', fontweight='bold')

from matplotlib.patches import Patch
legend_els = [Patch(color=v, label=f'Kelas {k+1}') for k, v in colors_map.items()]
plt.legend(handles=legend_els, loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('04_scatter_pred_vs_aktual.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.10 Ringkasan Hasil Evaluasi (Dashboard)

In [ ]:
# Rekalkulasi metrik per kelas
report = classification_report(y_test_0, y_pred, output_dict=True)
class_metrics = pd.DataFrame({
    'Precision': [report[str(i)]['precision'] for i in range(5)],
    'Recall'   : [report[str(i)]['recall']    for i in range(5)],
    'F1-Score' : [report[str(i)]['f1-score']  for i in range(5)]
}, index=[f'Kelas {i+1}' for i in range(5)])

fig = plt.figure(figsize=(18, 8))
gs = gridspec.GridSpec(1, 3, figure=fig)

# --- Subplot 1: Metrik per kelas (grouped bar)
ax1 = fig.add_subplot(gs[0, 0])
x = np.arange(5)
w = 0.25
ax1.bar(x - w, class_metrics['Precision'], w, label='Precision', color='#3498db', edgecolor='black')
ax1.bar(x,     class_metrics['Recall'],    w, label='Recall',    color='#2ecc71', edgecolor='black')
ax1.bar(x + w, class_metrics['F1-Score'],  w, label='F1-Score',  color='#e74c3c', edgecolor='black')
ax1.set_xticks(x)
ax1.set_xticklabels([f'Kelas {i+1}' for i in range(5)], rotation=30)
ax1.set_ylim(0.7, 1.05)
ax1.set_title('Metrik per Kelas', fontweight='bold')
ax1.legend(loc='lower right', fontsize=9)
ax1.set_ylabel('Score')

# --- Subplot 2: Overall Metrics (Gauge-style bar)
ax2 = fig.add_subplot(gs[0, 1])
metrics  = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
values   = [acc, prec, rec, f1, auc]
bar_cols = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#8e44ad']
bars2 = ax2.bar(metrics, values, color=bar_cols, edgecolor='black')
ax2.set_ylim(0.8, 1.05)
ax2.set_title('Overall Model Performance', fontweight='bold')
ax2.set_ylabel('Score')
ax2.tick_params(axis='x', rotation=20)
for bar, v in zip(bars2, values):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.005,
             f'{v:.3f}', ha='center', fontweight='bold', fontsize=10)

# --- Subplot 3: Probability Distribution for correct vs incorrect
ax3 = fig.add_subplot(gs[0, 2])
correct   = y_pred == y_test_0.values
max_probs = y_pred_proba.max(axis=1)
ax3.hist(max_probs[correct],   bins=20, alpha=0.7, color='#2ecc71', label='Benar', edgecolor='black')
ax3.hist(max_probs[~correct],  bins=20, alpha=0.7, color='#e74c3c', label='Salah', edgecolor='black')
ax3.set_xlabel('Max Probability')
ax3.set_ylabel('Frekuensi')
ax3.set_title('Distribusi Confidence\n(Benar vs Salah)', fontweight='bold')
ax3.legend()

plt.suptitle('Dashboard Evaluasi – XGBoost Smartphone Addiction Detection',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('04_dashboard_evaluasi.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Semua visualisasi berhasil disimpan!')
print('\n=== RINGKASAN AKHIR ===')
print(f'  Accuracy  : {acc*100:.2f}%')
print(f'  Precision : {prec*100:.2f}%')
print(f'  Recall    : {rec*100:.2f}%')
print(f'  F1-Score  : {f1*100:.2f}%')
print(f'  ROC-AUC   : {auc*100:.2f}%')